# 01 · Quickstart

EmbeddingGemma 2 turns text, images, videos and audio into **vectors**: lists of 768 numbers.
Things that mean similar things get similar vectors, so you can search, label and group them.

This notebook embeds a few documents and searches them with a text query.

In [1]:
import os
from pathlib import Path

import embeddinggemma_engine as eg

if Path.cwd().name == "notebooks":
    os.chdir("..")  # run from the repo root, so paths stay short
IMAGES = Path("assets/images")
VIDEOS = Path("assets/videos")
AUDIO = Path("assets/audio")


def mistake(fn):
    """Run fn and print the error the engine raises for it."""
    try:
        fn()
    except eg.EngineError as error:
        print(f"{type(error).__name__}: {error}\n")

## 1. Create the engine
Nothing loads yet. The model (1.5 GB of GPU memory) loads the first time you embed something.

In [2]:
engine = eg.EmbeddingEngine()

## 2. Embed some documents
An `Item` is one thing to embed. A plain string is a text item.
A `title` is optional and helps when your documents have one.

In [3]:
documents = [
    eg.Item.text("Bloom the grounds for thirty seconds, then pour slowly in circles.", title="Pour-over coffee"),
    eg.Item.text("Giraffes eat acacia leaves that other animals cannot reach.", title="Giraffe feeding"),
    eg.Item.image(IMAGES / "wildlife.jpg"),
    eg.Item.video(VIDEOS / "04_coffee.mp4"),
]
names = ["coffee text", "giraffe text", "wildlife photo", "coffee video"]

doc_vectors = engine.embed_documents(documents, task="search")
doc_vectors.vectors.shape

(4, 768)

## 3. Search
Queries are embedded with `embed_queries`, using the **same task** as the documents.
`top_k` returns the best matches with a similarity score (higher is closer).

In [4]:
query_vectors = engine.embed_queries(["how do I make coffee?", "animals on the savanna"], task="search")

for query, hits in zip(["how do I make coffee?", "animals on the savanna"], eg.top_k(query_vectors, doc_vectors, k=2)):
    print(query)
    for hit in hits:
        print(f"   {hit.score:.2f}  {names[hit.index]}")

how do I make coffee?
   0.74  coffee text
   0.67  coffee video
animals on the savanna
   0.73  wildlife photo
   0.66  giraffe text


Text, photos and videos are all in the same space, so one text query finds all of them.

## 4. What you get back
An `Embeddings` object: the vectors plus a record of how they were made.
The engine uses that record to stop you comparing vectors that don't belong together.

In [5]:
print("vectors:", doc_vectors.vectors.shape)
print("task:   ", doc_vectors.task.value)
print("side:   ", doc_vectors.side)
print("tokens: ", [t.total for t in doc_vectors.token_counts])

vectors: (4, 768)
task:    search
side:    document
tokens:  [24, 21, 280, 1100]


## Next
- **02 · Tasks and prompts**: what `task="search"` and the query/document split actually do. Read this one next.
- 03 · Search, 04 · Labelling, 05 · Duplicates and clustering, 06 · Media settings